In [0]:
# =====================================================
# 1️⃣ Widgets
# =====================================================

import json
from pyspark.sql.functions import current_timestamp

start_time = spark.sql("SELECT current_timestamp()").collect()[0][0]

dbutils.widgets.text(
    "table_metadata",
    "{'CustomerProductID':'1', 'TableID': '1', 'SourceTablename': 'Facilities', 'SourceSchema': 'Core', 'CustomerName': 'BostonHealthSystem', 'CustomerCode': 'BOSHOSP', 'ExtractionType': 'Incremental', 'WatermarkColumn': 'LastModifiedDateTime', 'LastExtractWatermark': '1900-01-01 00:00:00', 'TargetSchema': 'Bronze', 'IsActive': 'True'}"
)

dbutils.widgets.text("CustomerCode_Source", "BOSHOSP")

dbutils.widgets.text(
    "run_id",
    "472519629468310"
)
run_id=dbutils.widgets.get("run_id")

# Parse JSON safely
table_metadata = json.loads(dbutils.widgets.get("table_metadata").replace("'", '"'))
CustomerCode_Source = (dbutils.widgets.get("CustomerCode_Source"))

print("Table Metadata:", table_metadata)
print("Table CustomerCode_Source:", CustomerCode_Source)
print(f"Run ID: {run_id}")


In [0]:
# =====================================================
# 2️⃣ Extract Variables
# =====================================================

CustomerProductID=int(table_metadata["CustomerProductID"])
TableID = int(table_metadata["TableID"])
SourceTablename = table_metadata["SourceTablename"]
SourceSchema = table_metadata["SourceSchema"].lower()
CustomerName = table_metadata["CustomerName"]
CustomerCode = table_metadata["CustomerCode"].lower()
ExtractionType = table_metadata["ExtractionType"]
WatermarkColumn = table_metadata["WatermarkColumn"]
LastExtractWatermark=table_metadata["LastExtractWatermark"]
TargetSchema=table_metadata["TargetSchema"].lower()
IsActive=table_metadata["IsActive"]


#load_type = table_parameters.get("load_type")
#watermark_column = table_parameters.get("watermark_column")

bronze_table_fqn = f"clinicalforge.{TargetSchema}.{CustomerCode}_{SourceTablename}"

print(f"Target Bronze Table: {bronze_table_fqn}")

In [0]:
# DBTITLE 1,Metadata Entry
# =====================================================
# Make and entry to audit table
# =====================================================
entry_exists = spark.sql(f"""
    SELECT 1
    FROM clinicalforge.metadata.pipelinerun
    WHERE RunID = {run_id} AND TableID = {TableID}
""").count() > 0

if entry_exists:
    spark.sql(f"""
        UPDATE clinicalforge.metadata.pipelinerun
        SET
            TargetTableName = '{SourceTablename}',
            HealthClientID = '{CustomerCode}',
            StartDateTime = TIMESTAMP('{start_time}'),
            EndDateTime = NULL,
            RunStatus = 'INPROGRESS',
            ErrorMessage = NULL,
            RecordsIngested = 0
        WHERE RunID = {run_id} AND TableID = {TableID}
    """)
else:
    spark.sql(f"""
        INSERT INTO clinicalforge.metadata.pipelinerun (
            RunID,
            CustomerProductID,
            TableID,
            HealthClientID,
            TargetTableName,
            StartDateTime,
            EndDateTime,
            RecordsIngested,
            RunStatus,
            ErrorMessage)
        VALUES (
            {run_id},
            {CustomerProductID},
            {TableID},
            '{CustomerCode}',
            '{SourceTablename}',
            TIMESTAMP('{start_time}'),
            NULL,  -- end time
            0,
            'INPROGRESS',
            NULL --Error Message
        )
    """)


In [0]:
# =====================================================
# 3️⃣ Get Last Watermark (For Filtering Only)
# =====================================================

last_watermark = None

if ExtractionType in ["Incremental", "FullRefresh"] and LastExtractWatermark:
    watermark_df = spark.sql(f"""
        SELECT LastExtractWatermark
        FROM clinicalforge.metadata.tableslist
        WHERE TableID = {TableID}
    """)
    
    if watermark_df.count() > 0:
        last_watermark = watermark_df.first()["LastExtractWatermark"]

print("Last Watermark:", last_watermark)

In [0]:
df_conn_details=spark.sql(f"""select CD.ConnectionName, CD.SecretKeyName, C.CustomerName, C.CustomerCode, CD.IsActive from  clinicalforge.metadata.connectiondetails as CD
left join clinicalforge.metadata.CustomerProduct as CP on CP.CustomerProductID=CD.CustomerProductID
left join clinicalforge.metadata.Customers as C on CP.CustomerID=C.CustomerID
where CD.CustomerProductID={CustomerProductID}
""")
display(df_conn_details)

In [0]:
from pyspark.sql.functions import lit
# =====================================================
# 4️⃣ Read Source
# =====================================================
ScopeName=df_conn_details.select('ConnectionName').collect()[0]["ConnectionName"]
ScopeKey=df_conn_details.select('SecretKeyName').collect()[0]["SecretKeyName"]
# DBTITLE 1,Read Source
try:

    # 🔐 Read connection JSON from secret
    secret_json = dbutils.secrets.get(
        scope=ScopeName,
        key=ScopeKey
    )

    config = json.loads(secret_json)

    jdbc_url = f"jdbc:sqlserver://{config['host']}:{config['port']};database={config['database']}"

    jdbc_properties = {
        "user": config["user"],
        "password": config["password"],
        "driver": config["driver"]
    }

        # Build query
    if ExtractionType in ["Incremental", "FullRefresh"] and last_watermark:
        query = f"""
        (SELECT * FROM {SourceSchema}.{SourceTablename}
        WHERE {WatermarkColumn} > '{last_watermark}') AS src
        """
    else:
        query = f"(SELECT * FROM {SourceSchema}.{SourceSchema}) AS src"

    source_df = spark.read.jdbc(
        url=jdbc_url,
        table=query,
        properties=jdbc_properties
    )
    
    #    raise ValueError("Unsupported source_system")

    # =====================================================
    # 5️⃣ Add insert_timestamp
    # =====================================================

    source_df = source_df.withColumn("insert_timestamp", current_timestamp())
    source_df = source_df.withColumn("HealthClientID", lit(CustomerCode))

    # =====================================================
    # 6️⃣ Write to Bronze (Append Only)
    # =====================================================

    source_df.write.format("delta").mode("overwrite").saveAsTable(bronze_table_fqn)

    records_read = source_df.count()

    print("Source → Bronze Load Completed Successfully.")
    print("Watermark will be updated after Silver load.")

except Exception as e:
    end_time = spark.sql("SELECT current_timestamp()").collect()[0][0]
    error_message = str(e)

    spark.sql(f"""
        UPDATE clinicalforge.metadata.pipelinerun
        SET
            EndDateTime = TIMESTAMP('{end_time}'),
            RunStatus = 'FAILED',
            ErrorMessage = {'NULL' if not ErrorMessage else "'" + ErrorMessage.replace("'", "") + "'"}
        WHERE TableID = {table_id} AND RunID = {run_id} 
    """)
    raise

In [0]:
%sql
---drop table clinicalforge.bronze.boshosp_facilities